# Phase 4 — Feature Engineering & Predictive Machine Learning
## NISR 2026 Big Data Hackathon · Track 1: Agricultural Productivity

This notebook constructs a comprehensive feature matrix from Phase 2 (survey indicators) and Phase 3 (weather, satellite NDVI, eSoko market prices) data. We benchmark multiple machine learning regression algorithms to predict crop yield (`Yield_Kg_Ha`), formulate a composite **Climate Vulnerability & Irrigation Priority Index**, and generate production-ready artifacts.

**Key Deliverables:**
1. **Executed Notebook**: `notebooks/03_ml_yield_models.ipynb`
2. **Saved Models**: `models/yield_predictor_rf.pkl` & `models/yield_predictor_best.pkl`
3. **Model Predictions & Priority Scores**: `data/df_master_phase4_predictions.csv`
4. **Visualizations**:
   - `data/plot_feature_importance.png`
   - `data/plot_actual_vs_predicted_yield.png`
   - `data/plot_irrigation_priority_index.png`
   - `data/plot_correlation_matrix.png`
   - `data/plot_vulnerability_drivers.png`

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os, pathlib

from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
    print("✅ XGBoost loaded successfully")
except ImportError:
    HAS_XGB = False
    print("⚠️ XGBoost not installed")

# Configure plotting aesthetics
sns.set_theme(style='whitegrid', font_scale=1.15)
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 200,
    'savefig.bbox': 'tight',
    'figure.figsize': (12, 6),
})

OUT = pathlib.Path('../data')
MODEL_DIR = pathlib.Path('../models')
MODEL_DIR.mkdir(exist_ok=True)
print("Libraries and output paths initialized.")

✅ XGBoost loaded successfully
Libraries and output paths initialized.


## Step A — Feature Matrix Preparation & Preprocessing

In [2]:
# Load Phase 3 master dataset
df = pd.read_csv(OUT / 'df_master_phase3.csv')
print(f"Dataset shape: {df.shape}")
print(f"Districts: {df['DISTRICT'].unique().tolist()}")
print(f"Crops: {df['CROP'].unique().tolist()}")
df.head()

Dataset shape: (36, 20)
Districts: ['BURERA', 'GATSIBO', 'MUSANZE', 'HUYE', 'RUSIZI', 'GASABO']
Crops: ['Irish Potato', 'Maize', 'Beans', 'Cassava', 'Rice', 'Amaranth']


,DISTRICT,CROP,Cultivated_Area_Ha,Harvested_Area_Ha,Yield_Kg_Ha,Production_MT,Post_Harvest_Loss_Pct,Fertilizer_Pct,Irrigation_Pct,Extension_Reach_Pct,Smart_Nkunganire_Reach_Pct,Poverty_Headcount_Rate_Pct,Mean_Farmgate_Price_RWF,Annual_Precipitation_mm,Mean_Temperature_C,Total_ET0_mm,Dry_Days_Count,Water_Deficit_mm,NDVI_Mean_SeasonA,NDVI_Mean_SeasonB
0,BURERA,Irish Potato,4357.3810,4357.3810,10514.800000,45816.989739,0.27082,82.98,0.0,68.21,55.04,21.79,487.50,1313.3,17.86,1387.15,170,73.85,0.5797,0.62
1,BURERA,Maize,9650.8730,9599.3900,2097.012000,20130.036023,0.34733,82.98,0.0,68.21,55.04,21.79,663.83,1313.3,17.86,1387.15,170,73.85,0.5797,0.62
2,BURERA,Beans,10501.6325,10501.6325,953.456715,10012.852029,0.50048,82.98,0.0,68.21,55.04,21.79,1150.00,1313.3,17.86,1387.15,170,73.85,0.5797,0.62
3,BURERA,Cassava,108.3850,65.0310,12890.890000,838.307468,0.05212,82.98,0.0,68.21,55.04,21.79,633.33,1313.3,17.86,1387.15,170,73.85,0.5797,0.62
4,BURERA,Rice,0.0000,0.0000,0.000000,0.000000,1.98389,82.98,0.0,68.21,55.04,21.79,1450.00,1313.3,17.86,1387.15,170,73.85,0.5797,0.62


In [3]:
# Feature Engineering Domain Transformations
# 1. Harvest Efficiency (Ratio of harvested to cultivated area)
df['Harvest_Efficiency'] = np.where(
    df['Cultivated_Area_Ha'] > 0,
    df['Harvested_Area_Ha'] / df['Cultivated_Area_Ha'],
    0.0
)

# 2. Moisture Shortage Index (Evapotranspiration deficit relative to annual rainfall)
df['Moisture_Shortage_Index'] = np.where(
    df['Annual_Precipitation_mm'] > 0,
    df['Water_Deficit_mm'] / df['Annual_Precipitation_mm'],
    0.0
)

# 3. NDVI Seasonal Dynamics (Delta between Season B and Season A vegetative health)
df['NDVI_Seasonal_Change'] = df['NDVI_Mean_SeasonB'] - df['NDVI_Mean_SeasonA']

# 4. Infrastructure Gap (Irrigation deficit percentage)
df['Infrastructure_Gap'] = 100.0 - df['Irrigation_Pct']

# 5. Price to Yield Ratio (Economic value per kg of yield intensity)
df['Price_Yield_Ratio'] = np.where(
    df['Yield_Kg_Ha'] > 0,
    df['Mean_Farmgate_Price_RWF'] / df['Yield_Kg_Ha'],
    0.0
)

print(f"Engineered features added. Total columns: {df.shape[1]}")
engineered_cols = ['Harvest_Efficiency', 'Moisture_Shortage_Index', 'NDVI_Seasonal_Change',
                   'Infrastructure_Gap', 'Price_Yield_Ratio']
print(df[engineered_cols].describe().round(4).to_string())

Engineered features added. Total columns: 25
       Harvest_Efficiency  Moisture_Shortage_Index  NDVI_Seasonal_Change  Infrastructure_Gap  Price_Yield_Ratio
count             36.0000                  36.0000               36.0000             36.0000            36.0000
mean               0.8034                   0.5007                0.0131             69.5917             0.3926
std                0.3607                   0.4484                0.0352             20.3770             0.5668
min                0.0000                   0.0063               -0.0449             49.1200             0.0000
25%                0.8796                   0.0562                0.0013             50.0000             0.0455
50%                0.9981                   0.4498                0.0075             63.3800             0.1342
75%                1.0000                   0.7929                0.0403             91.6700             0.4237
max                1.0039                   1.2493         

In [4]:
# Feature grouping for preprocessing pipeline
categorical_cols = ['DISTRICT', 'CROP']
target_col = 'Yield_Kg_Ha'

# Exclude target and total production to prevent data leakage (since Production = Area * Yield)
exclude_cols = {target_col, 'Production_MT'}
numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in exclude_cols]

print(f"Categorical features ({len(categorical_cols)}): {categorical_cols}")
print(f"Numeric features ({len(numeric_cols)}): {numeric_cols}")
print(f"Target variable: {target_col}")

X = df[categorical_cols + numeric_cols].copy()
y = df[target_col].copy()

print(f"\nX shape: {X.shape}, y shape: {y.shape}")
print(f"Yield range: [{y.min():,.1f}, {y.max():,.1f}] kg/ha (mean: {y.mean():,.1f} kg/ha)")

Categorical features (2): ['DISTRICT', 'CROP']
Numeric features (21): ['Cultivated_Area_Ha', 'Harvested_Area_Ha', 'Post_Harvest_Loss_Pct', 'Fertilizer_Pct', 'Irrigation_Pct', 'Extension_Reach_Pct', 'Smart_Nkunganire_Reach_Pct', 'Poverty_Headcount_Rate_Pct', 'Mean_Farmgate_Price_RWF', 'Annual_Precipitation_mm', 'Mean_Temperature_C', 'Total_ET0_mm', 'Dry_Days_Count', 'Water_Deficit_mm', 'NDVI_Mean_SeasonA', 'NDVI_Mean_SeasonB', 'Harvest_Efficiency', 'Moisture_Shortage_Index', 'NDVI_Seasonal_Change', 'Infrastructure_Gap', 'Price_Yield_Ratio']
Target variable: Yield_Kg_Ha

X shape: (36, 23), y shape: (36,)
Yield range: [0.0, 14,634.3] kg/ha (mean: 5,436.2 kg/ha)


In [5]:
# Preprocessing pipeline: One-Hot Encoding for categorical, StandardScaler for numeric
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'),
         categorical_cols),
        ('num', StandardScaler(), numeric_cols)
    ],
    remainder='drop'
)

# Fit-transform to verify design matrix
X_transformed = preprocessor.fit_transform(X)
cat_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols).tolist()
all_feature_names = cat_feature_names + numeric_cols
print(f"Transformed design matrix shape: {X_transformed.shape}")
print(f"Total features: {len(all_feature_names)} ({len(cat_feature_names)} categorical one-hot + {len(numeric_cols)} numeric)")

Transformed design matrix shape: (36, 31)
Total features: 31 (10 categorical one-hot + 21 numeric)


In [6]:
# Feature correlation heatmap
corr_cols = numeric_cols + [target_col]
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
cmap = sns.diverging_palette(240, 10, as_cmap=True)
sns.heatmap(corr_matrix, mask=mask, cmap=cmap, center=0, annot=True, fmt='.2f',
            linewidths=0.5, ax=ax, square=True, annot_kws={'size': 7})
ax.set_title('Feature Correlation Matrix with Crop Yield', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(OUT / 'plot_correlation_matrix.png')
plt.show()
print("Correlation matrix plot saved.")

Correlation matrix plot saved.


## Step B — Model 1: District Crop Yield Predictor (Regression)

### Cross-Validation Strategy: Leave-One-Out Cross-Validation (LOOCV)
For tabular agronomic data with 36 district-crop pairings (6 target districts × 6 strategic crops), standard train/test splits (e.g., 80/20) result in only 7 test samples, leading to high test variance and fold instability. 
**Leave-One-Out Cross-Validation (LOOCV)** trains on 35 samples and evaluates on the single held-out sample across all 36 iterations. This ensures every data point serves as an out-of-fold validation target, delivering an unbiased, rigorous assessment of generalization error.

In [7]:
# Define diverse candidate models tuned for tabular agronomic data
loo = LeaveOneOut()

models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression (tuned)': Ridge(alpha=0.5),
    'Random Forest Regressor': RandomForestRegressor(
        n_estimators=250, max_depth=5, min_samples_leaf=2, max_features=0.8,
        random_state=42, n_jobs=-1
    ),
    'Gradient Boosting Regressor': GradientBoostingRegressor(
        n_estimators=100, max_depth=3, learning_rate=0.05, subsample=0.85,
        random_state=42
    ),
}

if HAS_XGB:
    models['XGBoost Regressor (tuned)'] = XGBRegressor(
        n_estimators=100, max_depth=3, learning_rate=0.05,
        subsample=0.85, colsample_bytree=0.85,
        reg_alpha=0.2, reg_lambda=1.2,
        random_state=42, verbosity=0
    )

results = {}
predictions_dict = {}

print("Evaluating models with Leave-One-Out Cross-Validation (LOOCV)...")
print("-" * 75)

for name, model in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    y_pred_loo = np.zeros(len(y))

    for train_idx, test_idx in loo.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        pipe.fit(X_train, y_train)
        y_pred_loo[test_idx] = pipe.predict(X_test)

    r2 = r2_score(y, y_pred_loo)
    rmse = np.sqrt(mean_squared_error(y, y_pred_loo))
    mae = mean_absolute_error(y, y_pred_loo)

    results[name] = {'R2': r2, 'RMSE': rmse, 'MAE': mae}
    predictions_dict[name] = y_pred_loo

    print(f"{name:30s} | R2={r2:+.4f} | RMSE={rmse:8,.1f} kg/ha | MAE={mae:8,.1f} kg/ha")

results_df = pd.DataFrame(results).T.sort_values('R2', ascending=False)
print("=" * 75)
print("FINAL BENCHMARK COMPARISON (LOOCV):")
print(results_df.round(4).to_string())

Evaluating models with Leave-One-Out Cross-Validation (LOOCV)...
---------------------------------------------------------------------------


Linear Regression              | R2=+0.6900 | RMSE= 2,463.2 kg/ha | MAE= 1,832.7 kg/ha


Ridge Regression (tuned)       | R2=+0.7307 | RMSE= 2,295.8 kg/ha | MAE= 1,740.8 kg/ha


Random Forest Regressor        | R2=+0.7088 | RMSE= 2,387.4 kg/ha | MAE= 1,748.2 kg/ha


Gradient Boosting Regressor    | R2=+0.8116 | RMSE= 1,920.4 kg/ha | MAE= 1,222.4 kg/ha


XGBoost Regressor (tuned)      | R2=+0.7965 | RMSE= 1,995.4 kg/ha | MAE= 1,458.9 kg/ha
FINAL BENCHMARK COMPARISON (LOOCV):
                                 R2       RMSE        MAE
Gradient Boosting Regressor  0.8116  1920.4314  1222.4105
XGBoost Regressor (tuned)    0.7965  1995.4311  1458.9189
Ridge Regression (tuned)     0.7307  2295.8398  1740.8178
Random Forest Regressor      0.7088  2387.4218  1748.2404
Linear Regression            0.6900  2463.2351  1832.7352


In [8]:
# Identify top performer
best_model_name = results_df.index[0]
print(f"Top Performing Model: {best_model_name} (LOOCV R2 = {results_df.loc[best_model_name, 'R2']:.4f})")

# Fit final pipelines on full dataset for production inference
# 1. Best overall model
final_best_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', models[best_model_name])
])
final_best_pipe.fit(X, y)
joblib.dump(final_best_pipe, MODEL_DIR / 'yield_predictor_best.pkl')
print(f"Saved best model pipeline to: {MODEL_DIR / 'yield_predictor_best.pkl'}")

# 2. Random Forest pipeline (to satisfy deliverable specification)
final_rf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', models['Random Forest Regressor'])
])
final_rf_pipe.fit(X, y)
joblib.dump(final_rf_pipe, MODEL_DIR / 'yield_predictor_rf.pkl')
print(f"Saved Random Forest pipeline to: {MODEL_DIR / 'yield_predictor_rf.pkl'}")

Top Performing Model: Gradient Boosting Regressor (LOOCV R2 = 0.8116)
Saved best model pipeline to: ..\models\yield_predictor_best.pkl


Saved Random Forest pipeline to: ..\models\yield_predictor_rf.pkl


In [9]:
# Feature importance extraction from tree ensembles
# We train tree regressors on the transformed feature matrix
rf_model = RandomForestRegressor(n_estimators=300, max_depth=5, min_samples_leaf=2, random_state=42)
rf_model.fit(X_transformed, y)

importance_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Importance_RF': rf_model.feature_importances_
})

if HAS_XGB:
    xgb_imp = XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42, verbosity=0)
    xgb_imp.fit(X_transformed, y)
    importance_df['Importance_XGB'] = xgb_imp.feature_importances_
    importance_df['Composite_Importance'] = (importance_df['Importance_RF'] + importance_df['Importance_XGB']) / 2
else:
    importance_df['Composite_Importance'] = importance_df['Importance_RF']

importance_df = importance_df.sort_values('Composite_Importance', ascending=True)

# Plot top predictors
fig, ax = plt.subplots(figsize=(11, 8))
top_n = min(15, len(importance_df))
top_features = importance_df.tail(top_n)

colors = plt.cm.viridis(np.linspace(0.3, 0.9, top_n))
bars = ax.barh(range(top_n), top_features['Composite_Importance'], color=colors, edgecolor='white', linewidth=0.5)
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_features['Feature'], fontsize=10)
ax.set_xlabel('Relative Feature Importance (MDI / Gain)', fontsize=12)
ax.set_title('Top Drivers of Crop Yield — Ensemble Feature Importance', fontsize=14, fontweight='bold', pad=12)

for bar, val in zip(bars, top_features['Composite_Importance']):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9, fontweight='bold')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(OUT / 'plot_feature_importance.png')
plt.show()
print("Feature importance visualization saved.")

Feature importance visualization saved.


In [10]:
# Actual vs Predicted diagnostic comparison using LOOCV predictions
best_loo_preds = predictions_dict[best_model_name]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left panel: Scatter plot of Actual vs Predicted
ax1 = axes[0]
crops = df['CROP'].unique()
palette = sns.color_palette('tab10', len(crops))
crop_colors = dict(zip(crops, palette))

for crop in crops:
    mask = df['CROP'] == crop
    ax1.scatter(y[mask], best_loo_preds[mask],
                c=[crop_colors[crop]], label=crop, s=90, alpha=0.85,
                edgecolors='white', linewidth=0.6)

min_val = min(y.min(), best_loo_preds.min()) * 0.9
max_val = max(y.max(), best_loo_preds.max()) * 1.08
ax1.plot([min_val, max_val], [min_val, max_val], '--', color='crimson', alpha=0.7, label='Ideal 1:1 Line')
ax1.set_xlim(min_val, max_val)
ax1.set_ylim(min_val, max_val)
ax1.set_xlabel('Actual Survey Yield (kg/ha)', fontsize=12)
ax1.set_ylabel('LOOCV Predicted Yield (kg/ha)', fontsize=12)
ax1.set_title(f'Actual vs. Predicted Crop Yield\n({best_model_name} — LOOCV R² = {results[best_model_name]["R2"]:.3f})',
              fontsize=13, fontweight='bold')
ax1.legend(loc='upper left', fontsize=9, framealpha=0.9)

# Right panel: Residual distribution
ax2 = axes[1]
residuals = y.values - best_loo_preds

for crop in crops:
    mask = df['CROP'] == crop
    ax2.scatter(best_loo_preds[mask], residuals[mask],
                c=[crop_colors[crop]], label=crop, s=90, alpha=0.85,
                edgecolors='white', linewidth=0.6)

ax2.axhline(0, color='crimson', linestyle='--', alpha=0.7)
ax2.set_xlabel('LOOCV Predicted Yield (kg/ha)', fontsize=12)
ax2.set_ylabel('Residual Error (Actual - Predicted kg/ha)', fontsize=12)
ax2.set_title(f'Residual Analysis (LOOCV RMSE = {results[best_model_name]["RMSE"]:,.1f} kg/ha)',
              fontsize=13, fontweight='bold')
ax2.legend(loc='upper left', fontsize=9, framealpha=0.9)

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(OUT / 'plot_actual_vs_predicted_yield.png')
plt.show()
print("Actual vs Predicted plot saved.")

Actual vs Predicted plot saved.


## Step C — Model 2: Climate Vulnerability & Irrigation Priority Index

To support evidence-based agricultural planning, we formulate a composite **Climate Vulnerability Score** across the 6 target districts. This multi-criteria decision model incorporates:
1. **Moisture Shortage Stress (30%)**: Evaporative water deficit relative to annual rainfall.
2. **Socioeconomic Fragility (25%)**: EICV7 Poverty Headcount Rate.
3. **Infrastructure Deficit (25%)**: Plot irrigation gap (100 - Irrigation_Pct) and extension outreach gap.
4. **Productivity Yield Deficit (20%)**: Normalized crop yield shortfall relative to high-performing peers.

In [11]:
# District-level aggregation for climate vulnerability scoring
district_agg = df.groupby('DISTRICT').agg({
    'Yield_Kg_Ha': 'mean',
    'Production_MT': 'sum',
    'Water_Deficit_mm': 'first',
    'Annual_Precipitation_mm': 'first',
    'Poverty_Headcount_Rate_Pct': 'first',
    'Irrigation_Pct': 'first',
    'Extension_Reach_Pct': 'first',
    'Post_Harvest_Loss_Pct': 'mean',
    'Fertilizer_Pct': 'first',
    'NDVI_Mean_SeasonA': 'first',
    'NDVI_Mean_SeasonB': 'first',
    'Mean_Temperature_C': 'first',
    'Dry_Days_Count': 'first',
}).reset_index()

# Sub-indices calculation
# 1. Moisture Shortage Index
district_agg['Moisture_Shortage_Index'] = (
    district_agg['Water_Deficit_mm'] / district_agg['Annual_Precipitation_mm']
)

# 2. Socioeconomic Fragility
district_agg['Socio_Fragility_Norm'] = (
    district_agg['Poverty_Headcount_Rate_Pct'] / district_agg['Poverty_Headcount_Rate_Pct'].max()
)

# 3. Infrastructure Deficit
district_agg['Infra_Gap_Irrigation'] = (100.0 - district_agg['Irrigation_Pct']) / 100.0
district_agg['Infra_Gap_Extension'] = (100.0 - district_agg['Extension_Reach_Pct']) / 100.0
district_agg['Infrastructure_Gap_Norm'] = (
    0.6 * district_agg['Infra_Gap_Irrigation'] + 0.4 * district_agg['Infra_Gap_Extension']
)

# 4. Yield Deficit
district_agg['Yield_Deficit_Norm'] = 1.0 - (
    district_agg['Yield_Kg_Ha'] / district_agg['Yield_Kg_Ha'].max()
)

# Composite Climate Vulnerability Score
weights = {
    'Moisture': 0.30,
    'Socioeconomic': 0.25,
    'Infrastructure': 0.25,
    'Yield_Deficit': 0.20
}

district_agg['Climate_Vulnerability_Score'] = (
    weights['Moisture'] * (district_agg['Moisture_Shortage_Index'] / district_agg['Moisture_Shortage_Index'].max()) +
    weights['Socioeconomic'] * district_agg['Socio_Fragility_Norm'] +
    weights['Infrastructure'] * (district_agg['Infrastructure_Gap_Norm'] / district_agg['Infrastructure_Gap_Norm'].max()) +
    weights['Yield_Deficit'] * district_agg['Yield_Deficit_Norm']
)

# Rank 1 = highest urgency for intervention
district_agg['Vulnerability_Rank'] = district_agg['Climate_Vulnerability_Score'].rank(ascending=False).astype(int)
district_agg = district_agg.sort_values('Climate_Vulnerability_Score', ascending=False)

print("=" * 82)
print("              CLIMATE VULNERABILITY & IRRIGATION PRIORITY INDEX")
print("=" * 82)
display_cols = ['DISTRICT', 'Climate_Vulnerability_Score', 'Vulnerability_Rank',
                'Moisture_Shortage_Index', 'Poverty_Headcount_Rate_Pct',
                'Irrigation_Pct', 'Extension_Reach_Pct', 'Yield_Kg_Ha']
print(district_agg[display_cols].to_string(index=False))
print("=" * 82)

              CLIMATE VULNERABILITY & IRRIGATION PRIORITY INDEX
DISTRICT  Climate_Vulnerability_Score  Vulnerability_Rank  Moisture_Shortage_Index  Poverty_Headcount_Rate_Pct  Irrigation_Pct  Extension_Reach_Pct  Yield_Kg_Ha
 GATSIBO                     0.571717                   1                 1.249319                       18.41           44.68                69.89  5369.045033
 MUSANZE                     0.537874                   2                 0.649891                       20.98            8.33                68.21  4847.782864
  GASABO                     0.458741                   3                 0.792941                       11.06           28.56                57.57  5710.854114
  RUSIZI                     0.409019                   4                 0.006294                       44.16           50.00                64.14  5566.598902
  BURERA                     0.388028                   5                 0.056232                       21.79            0.00     

In [12]:
# Multi-component stacked bar chart
fig, ax = plt.subplots(figsize=(12, 7))

districts = district_agg['DISTRICT'].tolist()
x = np.arange(len(districts))
width = 0.55

components = {
    'Moisture Stress (30%)': weights['Moisture'] * (district_agg['Moisture_Shortage_Index'].values / district_agg['Moisture_Shortage_Index'].max()),
    'Socioeconomic Fragility (25%)': weights['Socioeconomic'] * district_agg['Socio_Fragility_Norm'].values,
    'Infrastructure Gap (25%)': weights['Infrastructure'] * (district_agg['Infrastructure_Gap_Norm'].values / district_agg['Infrastructure_Gap_Norm'].max()),
    'Yield Deficit (20%)': weights['Yield_Deficit'] * district_agg['Yield_Deficit_Norm'].values,
}

palette = ['#D9534F', '#F0AD4E', '#0275D8', '#5CB85C']
bottom = np.zeros(len(districts))

for (label, vals), color in zip(components.items(), palette):
    ax.bar(x, vals, width, bottom=bottom, label=label, color=color, edgecolor='white', linewidth=0.5)
    bottom += vals

for i, (d, score, rank) in enumerate(zip(districts,
        district_agg['Climate_Vulnerability_Score'],
        district_agg['Vulnerability_Rank'])):
    ax.text(i, score + 0.012, f'#{rank}', ha='center', va='bottom', fontweight='bold', fontsize=12)

ax.set_xticks(x)
ax.set_xticklabels(districts, fontsize=11, fontweight='bold')
ax.set_ylabel('Composite Vulnerability Score', fontsize=12)
ax.set_title('District Climate Vulnerability & Irrigation Priority Ranking\n(#1 = Most Urgent Intervention Needed)',
             fontsize=14, fontweight='bold', pad=15)
ax.legend(loc='upper right', fontsize=10, framealpha=0.95)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_ylim(0, bottom.max() * 1.15)

plt.tight_layout()
plt.savefig(OUT / 'plot_irrigation_priority_index.png')
plt.show()
print("Irrigation priority index plot saved.")

Irrigation priority index plot saved.


In [13]:
# Breakdown of agricultural vulnerability drivers for highest priority district
top_district = district_agg.iloc[0]['DISTRICT']
driver_data = pd.DataFrame({
    'Driver': [
        'Evaporative Moisture Shortage',
        'Poverty Headcount Rate',
        'Plot Irrigation Deficit',
        'Agricultural Extension Gap',
        'Post-Harvest Loss Level',
        'Inorganic Fertilizer Shortfall',
        'Annual Dry Day Frequency',
        'Vegetation Health Deficit (1 - NDVI)'
    ],
    'Risk_Factor': [
        district_agg.iloc[0]['Moisture_Shortage_Index'],
        district_agg.iloc[0]['Poverty_Headcount_Rate_Pct'] / 100.0,
        (100.0 - district_agg.iloc[0]['Irrigation_Pct']) / 100.0,
        (100.0 - district_agg.iloc[0]['Extension_Reach_Pct']) / 100.0,
        district_agg.iloc[0]['Post_Harvest_Loss_Pct'] / 100.0,
        (100.0 - district_agg.iloc[0]['Fertilizer_Pct']) / 100.0,
        district_agg.iloc[0]['Dry_Days_Count'] / 365.0,
        1.0 - district_agg.iloc[0]['NDVI_Mean_SeasonA'],
    ]
}).sort_values('Risk_Factor', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors_bar = plt.cm.RdYlGn_r(np.linspace(0.2, 0.9, len(driver_data)))
bars = ax.barh(driver_data['Driver'], driver_data['Risk_Factor'], color=colors_bar, edgecolor='white')

for bar, val in zip(bars, driver_data['Risk_Factor']):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Normalized Risk Exposure (0 to 1)', fontsize=12)
ax.set_title(f'Top Agricultural Vulnerability Drivers — {top_district} District (#1 Priority)',
             fontsize=13, fontweight='bold', pad=12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlim(0, 1.15)

plt.tight_layout()
plt.savefig(OUT / 'plot_vulnerability_drivers.png')
plt.show()
print("Vulnerability drivers plot saved.")

Vulnerability drivers plot saved.


## Step D — Model Export & Final Predictions Dataset

In [14]:
# Export complete predictions and risk flags to CSV
df_pred = df.copy()

# Add LOOCV predictions from top model
df_pred['Predicted_Yield_Kg_Ha'] = best_loo_preds
df_pred['Yield_Prediction_Error'] = df_pred['Yield_Kg_Ha'] - df_pred['Predicted_Yield_Kg_Ha']
df_pred['Yield_Prediction_Error_Pct'] = np.where(
    df_pred['Yield_Kg_Ha'] > 0,
    (df_pred['Yield_Prediction_Error'] / df_pred['Yield_Kg_Ha']) * 100.0,
    0.0
)

# Merge district vulnerability scores
vuln_merge = district_agg[['DISTRICT', 'Climate_Vulnerability_Score', 'Vulnerability_Rank']].copy()
df_pred = df_pred.merge(vuln_merge, on='DISTRICT', how='left')

# Categorical priority indicator
df_pred['Irrigation_Priority'] = df_pred['Vulnerability_Rank'].map(
    lambda r: 'URGENT' if r <= 2 else ('HIGH' if r <= 4 else 'MODERATE')
)

output_path = OUT / 'df_master_phase4_predictions.csv'
df_pred.to_csv(output_path, index=False)
print(f"Predictions and risk rankings exported to: {output_path}")
print(f"File dimensions: {df_pred.shape[0]} rows x {df_pred.shape[1]} columns")
df_pred[['DISTRICT', 'CROP', 'Yield_Kg_Ha', 'Predicted_Yield_Kg_Ha', 'Yield_Prediction_Error', 'Climate_Vulnerability_Score', 'Irrigation_Priority']].head(10)

Predictions and risk rankings exported to: ..\data\df_master_phase4_predictions.csv
File dimensions: 36 rows x 31 columns


,DISTRICT,CROP,Yield_Kg_Ha,Predicted_Yield_Kg_Ha,Yield_Prediction_Error,Climate_Vulnerability_Score,Irrigation_Priority
0,BURERA,Irish Potato,10514.800000,12057.152907,-1542.352907,0.388028,MODERATE
1,BURERA,Maize,2097.012000,2553.474399,-456.462399,0.388028,MODERATE
2,BURERA,Beans,953.456715,790.255049,163.201666,0.388028,MODERATE
3,BURERA,Cassava,12890.890000,6951.687708,5939.202292,0.388028,MODERATE
4,BURERA,Rice,0.000000,464.827053,-464.827053,0.388028,MODERATE
5,BURERA,Amaranth,7609.037032,8384.881614,-775.844582,0.388028,MODERATE
6,GATSIBO,Irish Potato,5399.259906,5003.406495,395.853411,0.571717,URGENT
7,GATSIBO,Maize,1841.372649,1690.983524,150.389125,0.571717,URGENT
8,GATSIBO,Beans,725.210464,1048.793529,-323.583065,0.571717,URGENT
9,GATSIBO,Cassava,12887.570000,13369.248428,-481.678428,0.571717,URGENT


In [15]:
# Final Executive Summary
print("=" * 80)
print("          NISR HACKATHON: PHASE 4 MODELING & VULNERABILITY SUMMARY")
print("=" * 80)
print(f"\n1. CROP YIELD PREDICTOR BENCHMARK (36 Observations, LOOCV Evaluation):")
print(results_df.round(4).to_string())
print(f"\n🏆 Champion Model: {best_model_name}")
print(f"   • Cross-Validated R²:   {results[best_model_name]['R2']:.4f}")
print(f"   • Cross-Validated RMSE: {results[best_model_name]['RMSE']:,.1f} kg/ha")
print(f"   • Cross-Validated MAE:  {results[best_model_name]['MAE']:,.1f} kg/ha")

print(f"\n{'=' * 80}")
print("2. DISTRICT CLIMATE VULNERABILITY & IRRIGATION PRIORITY RANKING:")
print("=" * 80)
for _, row in district_agg.iterrows():
    rank = int(row['Vulnerability_Rank'])
    level = '🔴 URGENT' if rank <= 2 else ('🟡 HIGH' if rank <= 4 else '🟢 MODERATE')
    print(f"  #{rank} {row['DISTRICT']:12s} | Score: {row['Climate_Vulnerability_Score']:.4f} | "
          f"Irrigation: {row['Irrigation_Pct']:5.1f}% | Poverty: {row['Poverty_Headcount_Rate_Pct']:5.1f}% | Priority: {level}")

print(f"\n{'=' * 80}")
print("3. ARTIFACTS VERIFICATION:")
print("=" * 80)
deliverables = {
    'Primary Model (RF)': MODEL_DIR / 'yield_predictor_rf.pkl',
    'Champion Model': MODEL_DIR / 'yield_predictor_best.pkl',
    'Predictions CSV': OUT / 'df_master_phase4_predictions.csv',
    'Feature Importance Plot': OUT / 'plot_feature_importance.png',
    'Actual vs Predicted Plot': OUT / 'plot_actual_vs_predicted_yield.png',
    'Irrigation Priority Plot': OUT / 'plot_irrigation_priority_index.png',
    'Correlation Matrix Plot': OUT / 'plot_correlation_matrix.png',
    'Vulnerability Drivers Plot': OUT / 'plot_vulnerability_drivers.png',
}
for name, path in deliverables.items():
    status = '✅ OK' if path.exists() else '❌ MISSING'
    print(f"  [{status}] {name:28s}: {path}")

print(f"\n{'=' * 80}")
print("Phase 4 successfully completed at maximum capacity.")

          NISR HACKATHON: PHASE 4 MODELING & VULNERABILITY SUMMARY

1. CROP YIELD PREDICTOR BENCHMARK (36 Observations, LOOCV Evaluation):
                                 R2       RMSE        MAE
Gradient Boosting Regressor  0.8116  1920.4314  1222.4105
XGBoost Regressor (tuned)    0.7965  1995.4311  1458.9189
Ridge Regression (tuned)     0.7307  2295.8398  1740.8178
Random Forest Regressor      0.7088  2387.4218  1748.2404
Linear Regression            0.6900  2463.2351  1832.7352

🏆 Champion Model: Gradient Boosting Regressor
   • Cross-Validated R²:   0.8116
   • Cross-Validated RMSE: 1,920.4 kg/ha
   • Cross-Validated MAE:  1,222.4 kg/ha

2. DISTRICT CLIMATE VULNERABILITY & IRRIGATION PRIORITY RANKING:
  #1 GATSIBO      | Score: 0.5717 | Irrigation:  44.7% | Poverty:  18.4% | Priority: 🔴 URGENT
  #2 MUSANZE      | Score: 0.5379 | Irrigation:   8.3% | Poverty:  21.0% | Priority: 🔴 URGENT
  #3 GASABO       | Score: 0.4587 | Irrigation:  28.6% | Poverty:  11.1% | Priority: 🟡 HIGH
  #4